# Self-play checkpoint evaluation

Evaluate checkpoints on one fixed set of 1,000 states after six turn-based throws. The first 500 states use the model on turns 7 and 9; the other 500 use it on turn 8.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / 'data_generation.py').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('Could not find repository root')
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import evaluation
import self_play_training as self_play

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
# Evaluation parameters.
seed = 2026
num_states = 1000
checkpoint_indices = (1, 2, 5, 10, 20, 30)
grid_size = (10, 10, 4)
state_generator_version = 'turn_based'
max_stones = 10

checkpoint_dir = repo_root / 'weights' / 'self_play' / 'round9_20k'
evaluation_dir = Path('/home/vietafan/curling_data/self_play_N5_round9_20k/evaluation')
evaluation_dir.mkdir(parents=True, exist_ok=True)
states_path = evaluation_dir / 'states_after_6_throws.npz'
if states_path.exists():
    states = evaluation.load_sheet_states(states_path)
else:
    states = self_play.seq.data_generation.generate_random_sheet_state_for_turn(
        turn=6, num_sims=num_states, rng=__import__('numpy').random.default_rng(seed)
    )
    evaluation.write_sheet_states(states_path, states)
print(f'evaluation states: {states.x.shape[0]}')

evaluation states: 1000


In [3]:
records = self_play.evaluate_checkpoints(
    checkpoint_dir=checkpoint_dir,
    checkpoint_indices=checkpoint_indices,
    num_states=num_states,
    seed=seed,
    grid_size=grid_size,
    state_generator_version=state_generator_version,
    max_stones=max_stones,
    states=states,
)
records

[{'checkpoint': 1,
  'group': 'model_turns_7_9',
  'model_team': 0,
  'num_states': 500,
  'average_score': -1.06,
  'standard_error': 0.010631371130019326},
 {'checkpoint': 1,
  'group': 'model_turn_8',
  'model_team': 1,
  'num_states': 500,
  'average_score': 2.254,
  'standard_error': 0.03781306196687358},
 {'checkpoint': 2,
  'group': 'model_turns_7_9',
  'model_team': 0,
  'num_states': 500,
  'average_score': -1.182,
  'standard_error': 0.017272773297730432},
 {'checkpoint': 2,
  'group': 'model_turn_8',
  'model_team': 1,
  'num_states': 500,
  'average_score': 2.248,
  'standard_error': 0.040301866966207946},
 {'checkpoint': 5,
  'group': 'model_turns_7_9',
  'model_team': 0,
  'num_states': 500,
  'average_score': -1.0,
  'standard_error': 0.004903885825790071},
 {'checkpoint': 5,
  'group': 'model_turn_8',
  'model_team': 1,
  'num_states': 500,
  'average_score': 2.122,
  'standard_error': 0.0394124178065075},
 {'checkpoint': 10,
  'group': 'model_turns_7_9',
  'model_team'

In [ ]:
import json
results_path = evaluation_dir / 'checkpoint_scores.json'
results_path.write_text(json.dumps(records, indent=2))
for record in records:
    print(
        f"m_{record['checkpoint']:>2} {record['group']:<16} "
        f"mean={record['average_score']:.4f} +/- "
        f"{record['standard_error']:.4f}"
    )